# Plant trait data preprocessing and dataset preparation for BHPMF/EBPMF modeling.
Preprocesses and merges plant species descriptions with trait data, standardizes species and trait names, and generates randomized train/validation/test splits for downstream trait prediction and BHPMF/EBPMF modeling.

In [1]:
# =============================================================================
# Data preparation (refactored)
#
# Key change: ONE master table, all arrays extracted from it by column name.
# Eliminates the separate flora_merged_table merge and positional iloc slicing.
# Output variables are identical to the original notebook cells 1–8.
# =============================================================================

import os, random
import numpy as np
import pandas as pd
import torch

from scipy.stats import pearsonr
from sklearn.metrics import r2_score
from torch.utils.data import TensorDataset, Subset, DataLoader

import sys
sys.path.append('python')
sys.path.append('data')

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'


# =========================================================
# 1. Reproducibility
# =========================================================

def set_seed(seed=520):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

set_seed(520)


# =========================================================
# 2. Load raw data
# =========================================================

dess         = pd.read_csv('data/floras_0609_clean.csv').iloc[:, 1:]
flora_traits = pd.read_excel('data/appendix4_flora_traits.xlsx')
traits_all   = pd.read_csv('data/species_all_traits_mean.csv')
list_pd      = pd.read_csv('data/selected_traits4.csv')
all_taxa     = pd.read_csv('data/flora_with_higher_taxaall.csv')
embs         = np.load(
    'data/flora_emb_multilingual_0617_with_metadata.npz',
    allow_pickle=True
)['embeddings']


# =========================================================
# 3. Trait name mapping (for display only)
# =========================================================

trait_name_clean = {
    'Plant height vegetative':          'Vegetative height',
    'Seed length':                      'Seed length',
    'Bark thickness':                   'Bark thickness',
    'Stem diameter':                    'Stem diameter',
    'Seed dry mass':                    'Seed mass',
    'Fruit length':                     'Fruit length',
    'Dispersal unit length':            'Dispersal length',
    'Leaf area per leaf dry mass (specific leaf area, SLA or 1/LMA): petiole excluded': 'SLA',
    'Leaf dry mass per leaf fresh mass (leaf dry matter content, LDMC)':                 'LDMC',
    'Leaf density (leaf tissue density, leaf dry mass per leaf volume)':                 'Leaf density',
    'Leaf fresh mass':                  'Leaf fresh mass',
    'Leaf length':                      'Leaf length',
    'Leaf width':                       'Leaf width',
    'Leaf area (in case of compound leaves: leaflet, petiole excluded)': 'Leaf area',
    'Leaf nitrogen (N) content per leaf dry mass':      'Leaf N',
    'Leaf potassium (K) content per leaf dry mass':     'Leaf K',
    'Leaf phosphorus (P) content per leaf dry mass':    'Leaf P',
    'Leaf carbon (C) content per leaf dry mass':        'Leaf C',
    'Leaf carbon isotope signature (delta 13C)':        'Leaf δ13C',
    'Leaf calcium (Ca) content per leaf dry mass':      'Leaf Ca',
    'Leaf nitrogen (N) isotope signature (delta 15N)':  'Leaf δ15N',
    'Leaf water content per leaf dry mass (not saturated)':      'Leaf water',
    'Leaf chlorophyll content per leaf dry mass':                 'Chlorophyll',
    'Leaf respiration rate mass':                                 'Respiration',
    'Leaf carotenoid content per leaf dry mass':                  'Carotenoids',
    'Leaf transpiration rate per leaf area':                      'Leaf Transpiration',
    'Vcmax':                            'Vcmax',
    'Photosynthesis rate per leaf dry mass':             'Photosynthesis',
    'Stem specific density (SSD, stem dry mass per stem fresh volume) or wood density':   'SSD',
    'Stem conduit density (vessels and tracheids)':      'Conduit density',
    'Wood vessel diameter':             'Vessel diameter',
    'Wood vessel density':              'Vessel density',
    'Stem conduit cross-sectional area (vessels and tracheids)':  'Conduit area',
    'Stem conduit lumen cross-sectional area (vessels and tracheids) per stem sapwood cross-sectional area': 'Lumen fraction',
    'Wood cross-sectional fraction of fibre area':       'Fibre fraction',
    'Root rooting depth':               'Root depth',
    'Fine root (absorptive) length per absorptive fine root dry mass (specific absorptive fine root length, SRL)': 'Fine-root SRL',
    'Fine root diameter':               'Fine-root diameter',
    'Fine root tissue density (fine root dry mass per fine root volume)': 'Root density',
    'Root nitrogen (N) content per root dry mass':       'Root N',
    'Fine root nitrogen (N) content per fine root dry mass':      'Fine-root N',
    'Root length per root dry mass (specific root length, SRL)':  'SRL',
}

list_pd['Trait_formal'] = list_pd['Trait'].replace(trait_name_clean)
names = list_pd['Trait_formal'].values


# =========================================================
# 4. Clean traits_all
# =========================================================

traits_all.rename(columns={'Unnamed: 0': 'species'}, inplace=True)
traits_all['species'] = (
    traits_all['species'].str.lower().str.replace(' ', '_')
)
traits_all.iloc[:, 1:] = traits_all.iloc[:, 1:].apply(
    pd.to_numeric, errors='coerce'
)
traits_all = traits_all.dropna(axis=1, how='all')
traits_all.rename(columns={
    'Leaf respiration rate in the dark per leaf dry mass':
        'Leaf respiration rate mass',
    'Leaf respiration rate in the dark as fraction of '
    'photosynthetic carboxylation capacity (Vcmax)':
        'Vcmax',
}, inplace=True)

selected_traits = list_pd.Trait.values.tolist()
traits_all = traits_all[['species'] + selected_traits]


# =========================================================
# 5. Build ONE master table
# =========================================================
# dess and flora_traits share the same scientificName in the
# same order; dess only contributes re_index (embedding lookup).

flora_traits['_orig_idx'] = np.arange(len(flora_traits))

master = pd.merge(
    flora_traits, traits_all,
    left_on='scientificName', right_on='species',
    how='inner',
).merge(
    all_taxa[['scientificName', 'order', 'family', 'genus']],
    on='scientificName',
    how='left',
)

del dess                         # no longer needed


# =========================================================
# 6. Shuffle & split
# =========================================================

indexs = np.arange(len(master))
np.random.shuffle(indexs)

train_split = int(len(master) / 10 * 8)
val_split   = int(len(master) / 10 * 9)

train_indexes = range(0, train_split)
dev_indexes   = range(train_split, val_split)
test_indexes  = range(val_split, len(master))


# =========================================================
# 7. Extract aligned arrays — all from master[indexs]
# =========================================================

# --- target traits ---
f_traits_cal = master[selected_traits].astype(float).values[indexs]

# --- taxonomy ---
taxa_info = master[['order', 'family', 'genus']].iloc[indexs].reset_index(
    drop=True
)

# --- text embeddings (map back to original flora_traits row) ---
orig_idx = master['_orig_idx'].values
sentences = [embs[orig_idx[i]] for i in indexs]

# --- flora traits (28 numeric columns from flora_traits) ---
flora_feat_cols = (
    flora_traits.select_dtypes(include=[np.number])
    .columns.drop('_orig_idx')
    .tolist()
)
fdtraits = master[flora_feat_cols].astype(float).values[indexs]


# =========================================================
# 8. Process vad (percentile clip → log)
# =========================================================

vad = f_traits_cal.copy()

q1  = np.nanpercentile(vad, 1,  axis=0)
q99 = np.nanpercentile(vad, 99, axis=0)
vad = np.where((vad >= q1) & (vad <= q99), vad, np.nan)

col_min = np.nanmin(vad, axis=0, keepdims=True)
shift   = np.where(col_min <= 0, 1 - col_min, 0)
vad     = np.log(vad + shift)


# =========================================================
# 9. Random mask → trait_matrix, target, val_labels
# =========================================================

from sklearn.experimental import enable_iterative_imputer   # noqa
from sklearn.impute import IterativeImputer


def to_cuda(x):
    return torch.tensor(x, dtype=torch.float32).cuda()


def apply_random_mask(data, drop_prob=0.2, seed=520):
    orig_mask = torch.isnan(data)
    g = torch.Generator(device=data.device)
    g.manual_seed(seed)
    random_drop = (
        torch.rand(data.shape, device=data.device, generator=g) < drop_prob
    ) & (~orig_mask)

    input_data = data.clone()
    input_data[orig_mask]   = 0.0
    input_data[random_drop] = 0.0

    target = torch.full_like(data, float('nan'))
    target[random_drop] = data[random_drop]

    final_mask = ~orig_mask & ~random_drop
    return final_mask, input_data, target


def mice_impute(input_data):
    is_tensor = isinstance(input_data, torch.Tensor)
    device = input_data.device if is_tensor else None
    data_np = input_data.cpu().numpy() if is_tensor else input_data.copy()
    data_np = data_np.astype(float)
    data_np[data_np == 0] = np.nan
    orig_shape = data_np.shape
    flat = data_np.reshape(-1, orig_shape[-1])
    imp = IterativeImputer(random_state=0)
    filled = imp.fit_transform(flat).reshape(orig_shape)
    if is_tensor:
        return torch.tensor(filled, dtype=input_data.dtype).to(device)
    return filled


final_mask, input_data, target = apply_random_mask(
    to_cuda(vad), drop_prob=0.2
)

trait_matrix = input_data.detach().cpu().numpy()
trait_matrix[trait_matrix == 0] = np.nan

val_labels = target.detach().cpu().numpy()


# =========================================================
# 10. Process fdtraits (clip → MICE → log → z-score)
# =========================================================

fdtraits[fdtraits > 1e5] = np.nan

p2  = np.nanpercentile(fdtraits, 2,  axis=0)
p98 = np.nanpercentile(fdtraits, 98, axis=0)
fdtraits_clipped = np.clip(fdtraits, p2, p98)

fdtraits_reduced = mice_impute(fdtraits_clipped)

col_min = np.nanmin(fdtraits_reduced, axis=0, keepdims=True)
shift   = np.where(fdtraits_reduced <= 0, 1 - col_min, 0)
fdtraits_reduced_log = np.log(fdtraits_reduced + shift)

fd_mean = np.nanmean(fdtraits_reduced_log[:train_split], axis=0, keepdims=True)
fd_std  = np.nanstd(fdtraits_reduced_log[:train_split],  axis=0, keepdims=True)
fdtraits_reduced = (fdtraits_reduced_log - fd_mean) / fd_std

C:\Users\cml\AppData\Local\Temp\ipykernel_4344\1728937747.py:48: DtypeWarning: Columns (155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,179,180,181,182,183,184,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,227,228,232,235,236,237,240,241,242,244,245,246,252,254,255,256,257,258,259,260,261,262,263,264,266,267,268,269,270,271,272,273,275,276,277,278,279,280,281,282,296,298,299,300,318,319,320,321,322,323,325,326,327,328,329,331,332,333,335,336,337,338,339,340,341,342,343,344,345,346,348,349,350,351,352,353,355,356,357,358,359,360,361,362,364,371) have mixed types. Specify dtype option on import or set low_memory=False.
  traits_all   = pd.read_csv('data/species_all_traits_mean.csv')


In [4]:
# --- flora traits (28 numeric columns from flora_traits) ---
flora_feat_cols = (
    flora_traits.select_dtypes(include=[np.number])
    .columns.drop('_orig_idx')
    .tolist()
)
fdtraits = master[flora_feat_cols].astype(float).values[indexs]
fdtraits[fdtraits > 1e5] = np.nan

p2  = np.nanpercentile(fdtraits, 2,  axis=0)
p98 = np.nanpercentile(fdtraits, 98, axis=0)
fdtraits_clipped = np.clip(fdtraits, p2, p98)

fdtraits_reduced = mice_impute(fdtraits_clipped)

col_min = np.nanmin(fdtraits_reduced, axis=0, keepdims=True)
shift   = np.where(fdtraits_reduced <= 0, 1 - col_min, 0)
fdtraits_reduced_log = np.log(fdtraits_reduced + shift)

fd_mean = np.nanmean(fdtraits_reduced_log[:train_split], axis=0, keepdims=True)
fd_std  = np.nanstd(fdtraits_reduced_log[:train_split],  axis=0, keepdims=True)
fdtraits_reduced = (fdtraits_reduced_log - fd_mean) / fd_std

In [5]:
fdtraits

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], shape=(118383, 28))

In [6]:
print(np.isnan(fdtraits).mean())

nan_ratio = pd.Series(np.isnan(fdtraits).mean(axis=0), index=flora_feat_cols)
nan_ratio.sort_values(ascending=False)

0.8221354176094299


Bisexual: Tepal number                0.978299
Bisexual: Ovary cell number           0.976652
Chromosome number 2n                  0.976263
Bisexual: Carpel number               0.967918
Bisexual: Sepal/calyx number          0.952290
Anther number per flower              0.951108
Number of inflorescences per plant    0.950652
Stamen number                         0.947391
Seeds_per_fruit                       0.926560
Main stem length                      0.924896
Ovary length                          0.916001
Fruiting time                         0.906786
Elevation                             0.894588
Bisexual: Tepal length                0.883049
Bisexual: Stamen length               0.868300
Flower diameter                       0.854937
Seed length (cm)                      0.853543
Bisexual: Sepal/calyx width           0.851786
Flowering time (month)                0.783922
Fruit width (cm)                      0.781083
Bisexual: Petal length                0.729421
Leaf petiole 

# Training EBPMF
With the florstic embedding as the assistant information

In [ ]:
from bhpmf_torch import EmbeddingPriorBPMF_GPU
import pickle
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr


names = list_pd['Trait_formal'].values

# Normalize the trait matrix column-wise
col_mean = np.nanmean(trait_matrix, axis=0)
col_std = np.nanstd(trait_matrix, axis=0)
col_std[col_std < 1e-8] = 1.0
R_norm = (trait_matrix - col_mean) / col_std

# Run the model multiple times with different random seeds
n_runs = 10
all_predictions = []

for run_idx in range(n_runs):
    seed = 2026 + run_idx
    set_seed(seed)

    print(f'Running {run_idx + 1}/{n_runs}, seed={seed}')

    # Initialize EBMPF model with embedding prior
    model = EmbeddingPriorBPMF_GPU(
        n_latent=24,
        n_iterations=2000,
        burn_in=200,
        embed_precision=0.0001,
        learn_projection=True,
        w_freeze_after=999999,
        precision_anneal=False,
        device='cuda',
        verbose=True,
        max_samples=800,
        early_stop_patience=100,
        early_stop_min_samples=100,
    )

    # Fit the model using normalized traits and row embeddings
    model.fit(R_norm, row_embeddings=sentences)

    # Predict missing trait values using posterior samples
    pred_norm, _ = model.predict(
        return_std=True,
        n_samples=20,
        sample_gap=20
    )

    # Transform predictions back to the original scale
    predictions = pred_norm * col_std + col_mean
    all_predictions.append(predictions)

    # Evaluate predictions on randomly masked validation entries
    val_predictions = predictions
    test_list = []

    for i_index, i in enumerate(range(val_predictions.shape[1])):
        y_i = val_predictions[:, i]
        x_i = val_labels[:, i]

        # Keep only validation positions with observed target values
        mask = np.isnan(x_i)
        x_i = x_i[~mask]
        y_i = y_i[~mask]

        mse = mean_squared_error(x_i, y_i)

        # Min-max normalize values for scale-independent evaluation
        x_min = np.nanmin(vad[:, i_index])
        x_max = np.nanmax(vad[:, i_index])

        x_norm = (x_i - x_min) / (x_max - x_min + 1e-8)
        y_norm = (y_i - x_min) / (x_max - x_min + 1e-8)

        mse_norm = mean_squared_error(x_norm, y_norm)
        pearson_r, _ = pearsonr(x_norm, y_norm)

        test_list.append([
            names[i],
            pearson_r,
            mse,
            mse_norm,
            len(x_i)
        ])

    # Print average validation metrics for this run
    test_list_emb = pd.DataFrame(
        test_list,
        columns=['name', 'r2', 'mse', 'mse_norm', 'len']
    )

    print(test_list_emb.mean(numeric_only=True))

    if run_index == 0:
        pred_norm, _ = model.predict(
        return_std=True,
        n_samples=20,
        sample_gap=20
        )

    # Transform predictions back to the original scale
    predictions = pred_norm * col_std + col_mean 
    W = model.W_row_   # (n_latent, n_flora_traits)
    V = model.V_       # (n_target_traits, n_latent)'''
    # Save 
    np.save("outputs/W_emb_0706.npy", W)
    np.save("outputs/V_emb_0706.npy", V)
# Average predictions across all repeated runs
all_predictions_array = np.stack(all_predictions, axis=0)
predictions_mean = np.mean(all_predictions_array, axis=0)

# Save all run-level predictions
with open('outputs/0617_EBPMF_emb.pkl', 'wb') as f:
    pickle.dump(all_predictions_array, f)


# Final evaluation using the averaged predictions
val_predictions = predictions_mean
val_labels = target.detach().cpu().numpy()

test_list = []

for i_index, i in enumerate(range(val_predictions.shape[1])):
    y_i = val_predictions[:, i]
    x_i = val_labels[:, i]

    mask = np.isnan(x_i)
    x_i = x_i[~mask]
    y_i = y_i[~mask]

    mse = mean_squared_error(x_i, y_i)

    x_min = np.nanmin(vad[:, i_index])
    x_max = np.nanmax(vad[:, i_index])

    x_norm = (x_i - x_min) / (x_max - x_min + 1e-8)
    y_norm = (y_i - x_min) / (x_max - x_min + 1e-8)

    mse_norm = mean_squared_error(x_norm, y_norm)
    pearson_r, _ = pearsonr(x_norm, y_norm)

    test_list.append([
        names[i],
        pearson_r,
        mse,
        mse_norm,
        len(x_i)
    ])

test_list_emb = pd.DataFrame(
    test_list,
    columns=['name', 'r2', 'mse', 'mse_norm', 'len']
)

print(test_list_emb.mean(numeric_only=True))

Running 1/10, seed=2026
  Device: cuda
  Internal split: 110241 train / 12249 val (10% held out)
  row embedding: 1024d → PCA to 48d
  row embedding: (118383, 48)
  Iter 10/2000  train=0.6719  val=0.9855  alpha=2.04  [2.7 it/s, ETA 731s, 0 samples]
  Iter 20/2000  train=0.5076  val=1.0465  alpha=3.73  [2.9 it/s, ETA 678s, 0 samples]
  Iter 30/2000  train=0.4218  val=1.0941  alpha=5.40  [3.0 it/s, ETA 655s, 0 samples]
  Iter 40/2000  train=0.3647  val=1.1299  alpha=7.35  [3.0 it/s, ETA 644s, 0 samples]
  Iter 50/2000  train=0.3232  val=1.1497  alpha=9.37  [3.1 it/s, ETA 637s, 0 samples]
  Iter 60/2000  train=0.2929  val=1.1565  alpha=11.49  [3.1 it/s, ETA 633s, 0 samples]
  Iter 70/2000  train=0.2676  val=1.1781  alpha=13.87  [3.1 it/s, ETA 633s, 0 samples]
  Iter 80/2000  train=0.2490  val=1.1917  alpha=15.84  [3.0 it/s, ETA 631s, 0 samples]
  Iter 90/2000  train=0.2348  val=1.2019  alpha=17.85  [3.0 it/s, ETA 631s, 0 samples]
  Iter 100/2000  train=0.2207  val=1.2176  alpha=20.42  [3.

  Iter 910/2000  train=0.1387  val=1.0812  alpha=52.12  [2.7 it/s, ETA 398s, 710 samples]
  Iter 920/2000  train=0.1394  val=1.0732  alpha=51.31  [2.7 it/s, ETA 394s, 720 samples]
  Iter 930/2000  train=0.1392  val=1.0877  alpha=51.52  [2.7 it/s, ETA 391s, 730 samples]
  Iter 940/2000  train=0.1396  val=1.0924  alpha=51.62  [2.7 it/s, ETA 388s, 740 samples]
  Iter 950/2000  train=0.1423  val=1.0838  alpha=49.35  [2.7 it/s, ETA 384s, 750 samples]
  Iter 960/2000  train=0.1455  val=1.0716  alpha=47.40  [2.7 it/s, ETA 381s, 760 samples]
  Iter 970/2000  train=0.1457  val=1.0712  alpha=46.96  [2.7 it/s, ETA 377s, 770 samples]
  Iter 980/2000  train=0.1470  val=1.0775  alpha=46.23  [2.7 it/s, ETA 374s, 780 samples]
  Iter 990/2000  train=0.1499  val=1.0811  alpha=44.65  [2.7 it/s, ETA 370s, 790 samples]
  Iter 1000/2000  train=0.1514  val=1.0696  alpha=43.73  [2.7 it/s, ETA 367s, 800 samples]
  Iter 1010/2000  train=0.1568  val=1.0698  alpha=40.56  [2.7 it/s, ETA 363s, 800 samples]
  Iter 1

  Iter 410/2000  train=0.1041  val=1.2242  alpha=91.96  [3.1 it/s, ETA 517s, 210 samples]
  Iter 420/2000  train=0.1035  val=1.2247  alpha=93.73  [3.1 it/s, ETA 514s, 220 samples]
  Iter 430/2000  train=0.1054  val=1.2235  alpha=90.05  [3.1 it/s, ETA 511s, 230 samples]
  Iter 440/2000  train=0.1046  val=1.2025  alpha=90.52  [3.1 it/s, ETA 508s, 240 samples]
  Iter 450/2000  train=0.1027  val=1.2008  alpha=95.05  [3.1 it/s, ETA 506s, 250 samples]
  Iter 460/2000  train=0.1047  val=1.2150  alpha=90.96  [3.1 it/s, ETA 503s, 260 samples]
  Iter 470/2000  train=0.1045  val=1.1962  alpha=91.69  [3.1 it/s, ETA 501s, 270 samples]
  Iter 480/2000  train=0.1038  val=1.1884  alpha=92.25  [3.0 it/s, ETA 499s, 280 samples]
  Iter 490/2000  train=0.1024  val=1.1915  alpha=95.02  [3.0 it/s, ETA 497s, 290 samples]
  Iter 500/2000  train=0.1029  val=1.1955  alpha=94.72  [3.0 it/s, ETA 495s, 300 samples]
  Iter 510/2000  train=0.1020  val=1.1889  alpha=96.39  [3.0 it/s, ETA 493s, 310 samples]
  Iter 520

  Iter 70/2000  train=0.2719  val=1.1925  alpha=13.37  [3.0 it/s, ETA 647s, 0 samples]
  Iter 80/2000  train=0.2527  val=1.2093  alpha=15.58  [3.0 it/s, ETA 642s, 0 samples]
  Iter 90/2000  train=0.2371  val=1.2249  alpha=17.51  [3.0 it/s, ETA 638s, 0 samples]
  Iter 100/2000  train=0.2248  val=1.2310  alpha=19.44  [3.0 it/s, ETA 631s, 0 samples]
  Iter 110/2000  train=0.2160  val=1.2241  alpha=21.30  [3.0 it/s, ETA 627s, 0 samples]
  Iter 120/2000  train=0.2091  val=1.2429  alpha=22.84  [3.0 it/s, ETA 622s, 0 samples]
  Iter 130/2000  train=0.1965  val=1.2399  alpha=25.52  [3.0 it/s, ETA 617s, 0 samples]
  Iter 140/2000  train=0.1890  val=1.2520  alpha=27.79  [3.0 it/s, ETA 613s, 0 samples]
  Iter 150/2000  train=0.1791  val=1.2622  alpha=30.95  [3.0 it/s, ETA 608s, 0 samples]
  Iter 160/2000  train=0.1722  val=1.2883  alpha=33.36  [3.0 it/s, ETA 603s, 0 samples]
  Iter 170/2000  train=0.1689  val=1.2682  alpha=34.96  [3.0 it/s, ETA 600s, 0 samples]
  Iter 180/2000  train=0.1627  val=

  Iter 990/2000  train=0.1496  val=1.0765  alpha=44.84  [2.8 it/s, ETA 358s, 790 samples]
  Iter 1000/2000  train=0.1505  val=1.0646  alpha=44.28  [2.8 it/s, ETA 354s, 800 samples]
  Iter 1010/2000  train=0.1535  val=1.0736  alpha=42.66  [2.8 it/s, ETA 351s, 800 samples]
  Iter 1020/2000  train=0.1585  val=1.0601  alpha=39.98  [2.8 it/s, ETA 347s, 800 samples]
  Iter 1030/2000  train=0.1619  val=1.0674  alpha=38.15  [2.8 it/s, ETA 344s, 800 samples]
  Iter 1040/2000  train=0.1672  val=1.0775  alpha=36.23  [2.8 it/s, ETA 340s, 800 samples]
  Iter 1050/2000  train=0.1681  val=1.0524  alpha=35.37  [2.8 it/s, ETA 337s, 800 samples]
  Iter 1060/2000  train=0.1751  val=1.0548  alpha=32.71  [2.8 it/s, ETA 334s, 800 samples]
  Iter 1070/2000  train=0.1773  val=1.0755  alpha=32.03  [2.8 it/s, ETA 331s, 800 samples]
  Iter 1080/2000  train=0.1801  val=1.0717  alpha=30.77  [2.8 it/s, ETA 328s, 800 samples]
  Iter 1090/2000  train=0.1846  val=1.0588  alpha=29.22  [2.8 it/s, ETA 324s, 800 samples]


...

  Iter 290/2000  train=0.1269  val=1.2563  alpha=61.93  [3.1 it/s, ETA 552s, 90 samples]
  Iter 300/2000  train=0.1260  val=1.2548  alpha=62.83  [3.1 it/s, ETA 550s, 100 samples]
  Iter 310/2000  train=0.1221  val=1.2536  alpha=66.86  [3.1 it/s, ETA 548s, 110 samples]
  Iter 320/2000  train=0.1213  val=1.2566  alpha=67.69  [3.1 it/s, ETA 545s, 120 samples]
  Iter 330/2000  train=0.1202  val=1.2473  alpha=68.69  [3.1 it/s, ETA 543s, 130 samples]
  Iter 340/2000  train=0.1197  val=1.2523  alpha=69.76  [3.1 it/s, ETA 541s, 140 samples]
  Iter 350/2000  train=0.1180  val=1.2335  alpha=71.42  [3.1 it/s, ETA 538s, 150 samples]
  Iter 360/2000  train=0.1167  val=1.2398  alpha=73.52  [3.1 it/s, ETA 535s, 160 samples]
  Iter 370/2000  train=0.1152  val=1.2260  alpha=75.03  [3.1 it/s, ETA 531s, 170 samples]
  Iter 380/2000  train=0.1146  val=1.2265  alpha=75.82  [3.1 it/s, ETA 528s, 180 samples]
  Iter 390/2000  train=0.1126  val=1.2291  alpha=79.01  [3.1 it/s, ETA 525s, 190 samples]
  Iter 400/

  Iter 1200/2000  train=0.2232  val=1.0481  alpha=20.14  [2.7 it/s, ETA 291s, 800 samples]
  Iter 1210/2000  train=0.2266  val=1.0434  alpha=19.61  [2.7 it/s, ETA 288s, 800 samples]
  Iter 1220/2000  train=0.2251  val=1.0456  alpha=19.73  [2.7 it/s, ETA 285s, 800 samples]
  Iter 1230/2000  train=0.2288  val=1.0546  alpha=19.13  [2.7 it/s, ETA 282s, 800 samples]
  Iter 1240/2000  train=0.2320  val=1.0559  alpha=18.62  [2.7 it/s, ETA 279s, 800 samples]
  Iter 1250/2000  train=0.2345  val=1.0421  alpha=18.31  [2.7 it/s, ETA 275s, 800 samples]
  Iter 1260/2000  train=0.2387  val=1.0372  alpha=17.51  [2.7 it/s, ETA 272s, 800 samples]
  Iter 1270/2000  train=0.2409  val=1.0392  alpha=17.25  [2.7 it/s, ETA 269s, 800 samples]
  Iter 1280/2000  train=0.2423  val=1.0430  alpha=16.95  [2.7 it/s, ETA 265s, 800 samples]
  Iter 1290/2000  train=0.2459  val=1.0410  alpha=16.67  [2.7 it/s, ETA 262s, 800 samples]
  Iter 1300/2000  train=0.2446  val=1.0372  alpha=16.65  [2.7 it/s, ETA 258s, 800 samples]

  Iter 710/2000  train=0.1242  val=1.1032  alpha=64.98  [2.7 it/s, ETA 486s, 510 samples]
  Iter 720/2000  train=0.1273  val=1.1132  alpha=62.04  [2.7 it/s, ETA 483s, 520 samples]
  Iter 730/2000  train=0.1287  val=1.0926  alpha=60.77  [2.7 it/s, ETA 479s, 530 samples]
  Iter 740/2000  train=0.1291  val=1.1009  alpha=60.35  [2.7 it/s, ETA 475s, 540 samples]
  Iter 750/2000  train=0.1309  val=1.1009  alpha=58.77  [2.7 it/s, ETA 472s, 550 samples]
  Iter 760/2000  train=0.1320  val=1.1045  alpha=57.63  [2.6 it/s, ETA 468s, 560 samples]
  Iter 770/2000  train=0.1329  val=1.0969  alpha=56.66  [2.6 it/s, ETA 464s, 570 samples]
  Iter 780/2000  train=0.1336  val=1.0877  alpha=56.03  [2.6 it/s, ETA 461s, 580 samples]
  Iter 790/2000  train=0.1373  val=1.0832  alpha=53.40  [2.6 it/s, ETA 457s, 590 samples]
  Iter 800/2000  train=0.1380  val=1.0930  alpha=52.71  [2.6 it/s, ETA 453s, 600 samples]
  Iter 810/2000  train=0.1411  val=1.0892  alpha=50.45  [2.6 it/s, ETA 450s, 610 samples]
  Iter 820

  Iter 190/2000  train=0.1571  val=1.2767  alpha=40.26  [2.8 it/s, ETA 640s, 0 samples]
  Iter 200/2000  train=0.1507  val=1.2938  alpha=43.94  [2.8 it/s, ETA 635s, 0 samples]
  Iter 210/2000  train=0.1470  val=1.2902  alpha=46.03  [2.8 it/s, ETA 632s, 10 samples]
  Iter 220/2000  train=0.1465  val=1.2829  alpha=46.93  [2.8 it/s, ETA 629s, 20 samples]
  Iter 230/2000  train=0.1418  val=1.2872  alpha=49.54  [2.8 it/s, ETA 626s, 30 samples]
  Iter 240/2000  train=0.1399  val=1.2722  alpha=50.80  [2.8 it/s, ETA 622s, 40 samples]
  Iter 250/2000  train=0.1377  val=1.2775  alpha=52.38  [2.8 it/s, ETA 619s, 50 samples]
  Iter 260/2000  train=0.1350  val=1.2742  alpha=54.58  [2.8 it/s, ETA 616s, 60 samples]
  Iter 270/2000  train=0.1332  val=1.2718  alpha=56.21  [2.8 it/s, ETA 612s, 70 samples]
  Iter 280/2000  train=0.1308  val=1.2737  alpha=58.33  [2.8 it/s, ETA 608s, 80 samples]
  Iter 290/2000  train=0.1300  val=1.2575  alpha=59.09  [2.8 it/s, ETA 605s, 90 samples]
  Iter 300/2000  train=

  Iter 1110/2000  train=0.2155  val=1.0566  alpha=21.62  [2.6 it/s, ETA 336s, 800 samples]
  Iter 1120/2000  train=0.2208  val=1.0515  alpha=20.62  [2.6 it/s, ETA 332s, 800 samples]
  Iter 1130/2000  train=0.2229  val=1.0497  alpha=20.20  [2.6 it/s, ETA 329s, 800 samples]
  Iter 1140/2000  train=0.2281  val=1.0408  alpha=19.34  [2.6 it/s, ETA 326s, 800 samples]
  Iter 1150/2000  train=0.2295  val=1.0456  alpha=18.94  [2.6 it/s, ETA 322s, 800 samples]
  Iter 1160/2000  train=0.2317  val=1.0475  alpha=18.57  [2.6 it/s, ETA 318s, 800 samples]
  Iter 1170/2000  train=0.2335  val=1.0433  alpha=18.28  [2.6 it/s, ETA 314s, 800 samples]
  Iter 1180/2000  train=0.2363  val=1.0419  alpha=18.00  [2.6 it/s, ETA 310s, 800 samples]
  Iter 1190/2000  train=0.2384  val=1.0459  alpha=17.64  [2.6 it/s, ETA 307s, 800 samples]
  Iter 1200/2000  train=0.2379  val=1.0556  alpha=17.55  [2.6 it/s, ETA 303s, 800 samples]
  Iter 1210/2000  train=0.2416  val=1.0464  alpha=17.23  [2.6 it/s, ETA 299s, 800 samples]

# Training EBPMF
With the extracted traits from the flora as the assistant information

In [ ]:
from bhpmf_torch import EmbeddingPriorBPMF_GPU

# Normalize the trait matrix column-wise
col_mean = np.nanmean(trait_matrix, axis=0)
col_std = np.nanstd(trait_matrix, axis=0)
col_std[col_std < 1e-8] = 1.0
R_norm = (trait_matrix - col_mean) / col_std

# Run the model multiple times with different random seeds
n_runs = 10
all_predictions = []

for run_idx in range(n_runs):
    seed = 2026 + run_idx
    set_seed(seed)

    print(f'Running {run_idx + 1}/{n_runs}, seed={seed}')

    # Initialize EBMPF model with embedding prior
    model = EmbeddingPriorBPMF_GPU(
        n_latent=24,
        n_iterations=2000,
        burn_in=200,
        embed_precision=0.0001,
        learn_projection=True,
        w_freeze_after=999999,
        precision_anneal=False,
        device='cuda',
        verbose=True,
        max_samples=800,
        early_stop_patience=100,
        early_stop_min_samples=100,
    )

    # Fit the model using normalized traits and row embeddings
    model.fit(R_norm, row_embeddings=fdtraits_reduced)

    # Predict missing trait values using posterior samples
    pred_norm, _ = model.predict(
        return_std=True,
        n_samples=20,
        sample_gap=20
    )

    # Transform predictions back to the original scale
    predictions = pred_norm * col_std + col_mean
    all_predictions.append(predictions)

    # Evaluate predictions on randomly masked validation entries
    val_predictions = predictions
    test_list = []

    for i_index, i in enumerate(range(val_predictions.shape[1])):
        y_i = val_predictions[:, i]
        x_i = val_labels[:, i]

        # Keep only validation positions with observed target values
        mask = np.isnan(x_i)
        x_i = x_i[~mask]
        y_i = y_i[~mask]

        mse = mean_squared_error(x_i, y_i)

        # Min-max normalize values for scale-independent evaluation
        x_min = np.nanmin(vad[:, i_index])
        x_max = np.nanmax(vad[:, i_index])

        x_norm = (x_i - x_min) / (x_max - x_min + 1e-8)
        y_norm = (y_i - x_min) / (x_max - x_min + 1e-8)

        mse_norm = mean_squared_error(x_norm, y_norm)
        pearson_r, _ = pearsonr(x_norm, y_norm)

        test_list.append([
            names[i],
            pearson_r,
            mse,
            mse_norm,
            len(x_i)
        ])

    # Print average validation metrics for this run
    test_list_emb = pd.DataFrame(
        test_list,
        columns=['name', 'r2', 'mse', 'mse_norm', 'len']
    )

    print(test_list_emb.mean(numeric_only=True))

# Average predictions across all repeated runs
all_predictions_array = np.stack(all_predictions, axis=0)
predictions_mean = np.mean(all_predictions_array, axis=0)

# Save all run-level predictions
with open('outputs/0617_EBMPF_flora.pkl', 'wb') as f:
    pickle.dump(all_predictions_array, f)


# Final evaluation using the averaged predictions
val_predictions = predictions_mean
val_labels = target.detach().cpu().numpy()

test_list = []

for i_index, i in enumerate(range(val_predictions.shape[1])):
    y_i = val_predictions[:, i]
    x_i = val_labels[:, i]

    mask = np.isnan(x_i)
    x_i = x_i[~mask]
    y_i = y_i[~mask]

    mse = mean_squared_error(x_i, y_i)

    x_min = np.nanmin(vad[:, i_index])
    x_max = np.nanmax(vad[:, i_index])

    x_norm = (x_i - x_min) / (x_max - x_min + 1e-8)
    y_norm = (y_i - x_min) / (x_max - x_min + 1e-8)

    mse_norm = mean_squared_error(x_norm, y_norm)
    pearson_r, _ = pearsonr(x_norm, y_norm)

    test_list.append([
        names[i],
        pearson_r,
        mse,
        mse_norm,
        len(x_i)
    ])

test_list_emb = pd.DataFrame(
    test_list,
    columns=['name', 'r2', 'mse', 'mse_norm', 'len']
)

print(test_list_emb.mean(numeric_only=True))


Running 1/10, seed=2026
  Device: cuda
  Internal split: 110241 train / 12249 val (10% held out)
  row embedding: (118383, 28)
  Iter 10/2000  train=0.6941  val=0.9788  alpha=1.96  [3.1 it/s, ETA 649s, 0 samples]
  Iter 20/2000  train=0.5412  val=1.0424  alpha=3.30  [3.1 it/s, ETA 642s, 0 samples]
  Iter 30/2000  train=0.4601  val=1.0731  alpha=4.57  [3.2 it/s, ETA 623s, 0 samples]
  Iter 40/2000  train=0.4001  val=1.1105  alpha=6.15  [3.2 it/s, ETA 612s, 0 samples]
  Iter 50/2000  train=0.3582  val=1.1330  alpha=7.68  [3.2 it/s, ETA 606s, 0 samples]
  Iter 60/2000  train=0.3268  val=1.1456  alpha=9.21  [3.2 it/s, ETA 600s, 0 samples]
  Iter 70/2000  train=0.3071  val=1.1674  alpha=10.44  [3.2 it/s, ETA 596s, 0 samples]
  Iter 80/2000  train=0.2893  val=1.1740  alpha=11.80  [3.2 it/s, ETA 591s, 0 samples]
  Iter 90/2000  train=0.2774  val=1.1817  alpha=12.91  [3.3 it/s, ETA 584s, 0 samples]
  Iter 100/2000  train=0.2647  val=1.1759  alpha=14.20  [3.3 it/s, ETA 580s, 0 samples]
  Iter 1

  Iter 920/2000  train=0.2287  val=1.0494  alpha=19.03  [3.1 it/s, ETA 352s, 720 samples]
  Iter 930/2000  train=0.2269  val=1.0688  alpha=19.29  [3.1 it/s, ETA 349s, 730 samples]
  Iter 940/2000  train=0.2283  val=1.0720  alpha=19.25  [3.1 it/s, ETA 346s, 740 samples]
  Iter 950/2000  train=0.2285  val=1.0689  alpha=19.10  [3.1 it/s, ETA 343s, 750 samples]
  Iter 960/2000  train=0.2325  val=1.0748  alpha=18.44  [3.1 it/s, ETA 340s, 760 samples]
  Iter 970/2000  train=0.2319  val=1.0654  alpha=18.60  [3.1 it/s, ETA 337s, 770 samples]
  Iter 980/2000  train=0.2329  val=1.0643  alpha=18.59  [3.1 it/s, ETA 334s, 780 samples]
  Iter 990/2000  train=0.2392  val=1.0654  alpha=17.49  [3.1 it/s, ETA 331s, 790 samples]
  Iter 1000/2000  train=0.2371  val=1.0647  alpha=17.82  [3.1 it/s, ETA 328s, 800 samples]
  Iter 1010/2000  train=0.2380  val=1.0754  alpha=17.63  [3.1 it/s, ETA 325s, 800 samples]
  Iter 1020/2000  train=0.2421  val=1.0693  alpha=17.13  [3.1 it/s, ETA 321s, 800 samples]
  Iter 

  Iter 700/2000  train=0.1954  val=1.0871  alpha=26.21  [3.1 it/s, ETA 414s, 500 samples]
  Iter 710/2000  train=0.1984  val=1.0634  alpha=25.61  [3.1 it/s, ETA 412s, 510 samples]
  Iter 720/2000  train=0.2011  val=1.0805  alpha=24.81  [3.1 it/s, ETA 409s, 520 samples]
  Iter 730/2000  train=0.2056  val=1.0748  alpha=23.72  [3.1 it/s, ETA 406s, 530 samples]
  Iter 740/2000  train=0.2079  val=1.0775  alpha=22.97  [3.1 it/s, ETA 403s, 540 samples]
  Iter 750/2000  train=0.2073  val=1.0733  alpha=23.22  [3.1 it/s, ETA 400s, 550 samples]
  Iter 760/2000  train=0.2096  val=1.0749  alpha=22.88  [3.1 it/s, ETA 397s, 560 samples]
  Iter 770/2000  train=0.2122  val=1.0709  alpha=22.21  [3.1 it/s, ETA 394s, 570 samples]
  Iter 780/2000  train=0.2124  val=1.0644  alpha=22.01  [3.1 it/s, ETA 391s, 580 samples]
  Iter 790/2000  train=0.2126  val=1.0679  alpha=22.16  [3.1 it/s, ETA 388s, 590 samples]
  Iter 800/2000  train=0.2134  val=1.0742  alpha=21.94  [3.1 it/s, ETA 385s, 600 samples]
  Iter 810

  Iter 720/2000  train=0.1871  val=1.0897  alpha=28.43  [3.2 it/s, ETA 405s, 520 samples]
  Iter 730/2000  train=0.1899  val=1.0810  alpha=27.68  [3.2 it/s, ETA 403s, 530 samples]
  Iter 740/2000  train=0.1933  val=1.0914  alpha=26.83  [3.2 it/s, ETA 400s, 540 samples]
  Iter 750/2000  train=0.1928  val=1.0869  alpha=26.75  [3.1 it/s, ETA 397s, 550 samples]
  Iter 760/2000  train=0.1961  val=1.0709  alpha=25.95  [3.1 it/s, ETA 394s, 560 samples]
  Iter 770/2000  train=0.1983  val=1.0775  alpha=25.55  [3.1 it/s, ETA 392s, 570 samples]
  Iter 780/2000  train=0.2025  val=1.0795  alpha=24.30  [3.1 it/s, ETA 389s, 580 samples]
  Iter 790/2000  train=0.2047  val=1.0788  alpha=24.04  [3.1 it/s, ETA 386s, 590 samples]
  Iter 800/2000  train=0.2038  val=1.0808  alpha=24.11  [3.1 it/s, ETA 384s, 600 samples]
  Iter 810/2000  train=0.2072  val=1.0852  alpha=23.24  [3.1 it/s, ETA 381s, 610 samples]
  Iter 820/2000  train=0.2074  val=1.0645  alpha=23.25  [3.1 it/s, ETA 378s, 620 samples]
  Iter 830

  Iter 660/2000  train=0.1894  val=1.0876  alpha=27.72  [3.1 it/s, ETA 431s, 460 samples]
  Iter 670/2000  train=0.1897  val=1.0826  alpha=27.67  [3.1 it/s, ETA 428s, 470 samples]
  Iter 680/2000  train=0.1917  val=1.0795  alpha=27.20  [3.1 it/s, ETA 425s, 480 samples]
  Iter 690/2000  train=0.1955  val=1.0917  alpha=26.22  [3.1 it/s, ETA 422s, 490 samples]
  Iter 700/2000  train=0.1972  val=1.0876  alpha=25.62  [3.1 it/s, ETA 419s, 500 samples]
  Iter 710/2000  train=0.1986  val=1.0766  alpha=25.22  [3.1 it/s, ETA 416s, 510 samples]
  Iter 720/2000  train=0.2014  val=1.0735  alpha=24.78  [3.1 it/s, ETA 413s, 520 samples]
  Iter 730/2000  train=0.2033  val=1.0805  alpha=24.23  [3.1 it/s, ETA 411s, 530 samples]
  Iter 740/2000  train=0.2070  val=1.0711  alpha=23.50  [3.1 it/s, ETA 408s, 540 samples]
  Iter 750/2000  train=0.2058  val=1.0735  alpha=23.53  [3.1 it/s, ETA 404s, 550 samples]
  Iter 760/2000  train=0.2091  val=1.0829  alpha=22.81  [3.1 it/s, ETA 401s, 560 samples]
  Iter 770

...

  Iter 50/2000  train=0.3562  val=1.1136  alpha=7.72  [3.5 it/s, ETA 565s, 0 samples]
  Iter 60/2000  train=0.3251  val=1.1540  alpha=9.32  [3.5 it/s, ETA 560s, 0 samples]
  Iter 70/2000  train=0.3030  val=1.1569  alpha=10.72  [3.5 it/s, ETA 557s, 0 samples]
  Iter 80/2000  train=0.2811  val=1.1659  alpha=12.54  [3.5 it/s, ETA 554s, 0 samples]
  Iter 90/2000  train=0.2647  val=1.1801  alpha=14.14  [3.5 it/s, ETA 551s, 0 samples]
  Iter 100/2000  train=0.2533  val=1.2076  alpha=15.57  [3.5 it/s, ETA 548s, 0 samples]
  Iter 110/2000  train=0.2422  val=1.2038  alpha=16.94  [3.5 it/s, ETA 545s, 0 samples]
  Iter 120/2000  train=0.2338  val=1.2161  alpha=18.19  [3.5 it/s, ETA 543s, 0 samples]
  Iter 130/2000  train=0.2215  val=1.2121  alpha=20.17  [3.5 it/s, ETA 540s, 0 samples]
  Iter 140/2000  train=0.2147  val=1.2262  alpha=21.54  [3.5 it/s, ETA 537s, 0 samples]
  Iter 150/2000  train=0.2101  val=1.2160  alpha=22.51  [3.5 it/s, ETA 533s, 0 samples]
  Iter 160/2000  train=0.2071  val=1.22

  Iter 970/2000  train=0.2266  val=1.0562  alpha=19.50  [3.2 it/s, ETA 327s, 770 samples]
  Iter 980/2000  train=0.2286  val=1.0685  alpha=19.09  [3.2 it/s, ETA 324s, 780 samples]
  Iter 990/2000  train=0.2322  val=1.0730  alpha=18.68  [3.2 it/s, ETA 320s, 790 samples]
  Iter 1000/2000  train=0.2279  val=1.0696  alpha=19.22  [3.2 it/s, ETA 317s, 800 samples]
  Iter 1010/2000  train=0.2284  val=1.0695  alpha=19.23  [3.2 it/s, ETA 314s, 800 samples]
  Iter 1020/2000  train=0.2331  val=1.0687  alpha=18.32  [3.1 it/s, ETA 311s, 800 samples]
  Iter 1030/2000  train=0.2335  val=1.0749  alpha=18.38  [3.1 it/s, ETA 308s, 800 samples]
  Iter 1040/2000  train=0.2346  val=1.0625  alpha=18.09  [3.1 it/s, ETA 305s, 800 samples]
  Iter 1050/2000  train=0.2350  val=1.0784  alpha=18.07  [3.1 it/s, ETA 302s, 800 samples]
  Iter 1060/2000  train=0.2338  val=1.0658  alpha=18.28  [3.1 it/s, ETA 299s, 800 samples]
  Iter 1070/2000  train=0.2329  val=1.0754  alpha=18.46  [3.1 it/s, ETA 296s, 800 samples]
  

  Iter 770/2000  train=0.2104  val=1.0718  alpha=22.70  [3.2 it/s, ETA 388s, 570 samples]
  Iter 780/2000  train=0.2161  val=1.0706  alpha=21.37  [3.2 it/s, ETA 386s, 580 samples]
  Iter 790/2000  train=0.2198  val=1.0726  alpha=21.06  [3.2 it/s, ETA 383s, 590 samples]
  Iter 800/2000  train=0.2207  val=1.0710  alpha=20.48  [3.2 it/s, ETA 381s, 600 samples]
  Iter 810/2000  train=0.2249  val=1.0720  alpha=19.86  [3.2 it/s, ETA 378s, 610 samples]
  Iter 820/2000  train=0.2259  val=1.0654  alpha=19.70  [3.1 it/s, ETA 375s, 620 samples]
  Iter 830/2000  train=0.2218  val=1.0853  alpha=20.31  [3.1 it/s, ETA 372s, 630 samples]
  Iter 840/2000  train=0.2214  val=1.0700  alpha=20.50  [3.1 it/s, ETA 369s, 640 samples]
  Iter 850/2000  train=0.2230  val=1.0756  alpha=20.16  [3.1 it/s, ETA 367s, 650 samples]
  Iter 860/2000  train=0.2224  val=1.0792  alpha=20.21  [3.1 it/s, ETA 364s, 660 samples]
  Iter 870/2000  train=0.2203  val=1.0701  alpha=20.57  [3.1 it/s, ETA 361s, 670 samples]
  Iter 880

  Iter 530/2000  train=0.1667  val=1.1044  alpha=35.87  [3.3 it/s, ETA 447s, 330 samples]
  Iter 540/2000  train=0.1688  val=1.1085  alpha=35.30  [3.3 it/s, ETA 445s, 340 samples]
  Iter 550/2000  train=0.1689  val=1.1050  alpha=34.89  [3.3 it/s, ETA 443s, 350 samples]
  Iter 560/2000  train=0.1702  val=1.0986  alpha=34.22  [3.3 it/s, ETA 440s, 360 samples]
  Iter 570/2000  train=0.1748  val=1.1151  alpha=32.67  [3.2 it/s, ETA 440s, 370 samples]
  Iter 580/2000  train=0.1761  val=1.0940  alpha=32.08  [3.2 it/s, ETA 439s, 380 samples]
  Iter 590/2000  train=0.1774  val=1.1051  alpha=31.88  [3.2 it/s, ETA 436s, 390 samples]
  Iter 600/2000  train=0.1758  val=1.1026  alpha=32.33  [3.2 it/s, ETA 433s, 400 samples]
  Iter 610/2000  train=0.1746  val=1.0911  alpha=32.72  [3.2 it/s, ETA 430s, 410 samples]
  Iter 620/2000  train=0.1795  val=1.0933  alpha=31.09  [3.2 it/s, ETA 427s, 420 samples]
  Iter 630/2000  train=0.1827  val=1.0912  alpha=29.88  [3.2 it/s, ETA 424s, 430 samples]
  Iter 640

  Iter 380/2000  train=0.1667  val=1.1216  alpha=36.01  [3.4 it/s, ETA 482s, 180 samples]
  Iter 390/2000  train=0.1668  val=1.1238  alpha=35.71  [3.4 it/s, ETA 479s, 190 samples]
  Iter 400/2000  train=0.1653  val=1.1238  alpha=36.51  [3.4 it/s, ETA 476s, 200 samples]
  Iter 410/2000  train=0.1629  val=1.1102  alpha=37.47  [3.4 it/s, ETA 473s, 210 samples]
  Iter 420/2000  train=0.1632  val=1.0985  alpha=37.21  [3.4 it/s, ETA 471s, 220 samples]
  Iter 430/2000  train=0.1625  val=1.1151  alpha=37.79  [3.4 it/s, ETA 468s, 230 samples]
  Iter 440/2000  train=0.1619  val=1.1183  alpha=38.36  [3.3 it/s, ETA 466s, 240 samples]
  Iter 450/2000  train=0.1622  val=1.1165  alpha=38.14  [3.3 it/s, ETA 464s, 250 samples]
  Iter 460/2000  train=0.1650  val=1.0996  alpha=36.83  [3.3 it/s, ETA 462s, 260 samples]
  Iter 470/2000  train=0.1646  val=1.1092  alpha=37.10  [3.3 it/s, ETA 460s, 270 samples]
  Iter 480/2000  train=0.1630  val=1.0966  alpha=37.44  [3.3 it/s, ETA 458s, 280 samples]
  Iter 490

# MICE benchmark


In [15]:
from statsmodels.imputation.mice import MICEData

def mice_impute(input_data, n_iter=200, zero_as_missing=True):
    # Check whether the input is a PyTorch tensor
    is_tensor = isinstance(input_data, torch.Tensor)
    device = input_data.device if is_tensor else None

    # Convert input data to a NumPy array
    data_np = input_data.detach().cpu().numpy() if is_tensor else np.asarray(input_data)
    data_np = data_np.astype(float).copy()

    # Optionally treat zeros as missing values
    if zero_as_missing:
        data_np[data_np == 0] = np.nan

    orig_shape = data_np.shape
    flat = data_np.reshape(-1, orig_shape[-1])

    # Identify rows that are not entirely missing
    valid_rows = ~np.isnan(flat).all(axis=1)

    # Initialize the output matrix with the original values
    filled = flat.copy()

    # Apply MICE only to rows with at least one observed value
    df_valid = pd.DataFrame(
        flat[valid_rows],
        columns=[f"col_{i}" for i in range(flat.shape[1])]
    )

    mice_data = MICEData(df_valid)

    # Run MICE updates for the specified number of iterations
    for _ in range(n_iter):
        mice_data.update_all()

    filled[valid_rows] = mice_data.data.to_numpy()

    # Handle rows that are entirely missing.
    # Here, we use column means as a conservative fallback.
    col_means = np.nanmean(filled, axis=0)
    all_nan_rows = ~valid_rows

    if np.any(all_nan_rows):
        filled[all_nan_rows] = col_means

    filled = filled.reshape(orig_shape)

    # Convert back to a PyTorch tensor if the input was a tensor
    if is_tensor:
        return torch.tensor(filled, dtype=input_data.dtype, device=device)

    return filled

In [16]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error,accuracy_score,recall_score

col_mean = np.nanmean(trait_matrix, axis=0)
col_std = np.nanstd(trait_matrix, axis=0)
col_std[col_std < 1e-8] = 1.0
R_norm = (trait_matrix - col_mean) / col_std


val_predictions = mice_impute(R_norm)
val_labels = target.detach().cpu().numpy()#[test_indexes]
val_predictions = val_predictions * col_std + col_mean


test_list = []
for i_index,i in enumerate(range(val_predictions.shape[1])):
    y_i = val_predictions[:, i]
    x_i = val_labels[:, i]

    mask = np.isnan(x_i)
    x_i = x_i[~mask]
    y_i = y_i[~mask]

    # =====  MSE =====
    mse = mean_squared_error(x_i, y_i)

    # ===== Normlization (Min-Max) =====
    x_min, x_max = np.nanmin(vad[:,i_index]), np.nanmax(vad[:,i_index])
    y_min, y_max = np.nanmin(vad[:,i_index]), np.nanmax(vad[:,i_index])

    x_norm = (x_i - x_min) / (x_max - x_min + 1e-8)
    y_norm = (y_i - y_min) / (y_max - y_min + 1e-8)

    mse_norm = mean_squared_error(x_norm, y_norm)

    test_list.append([
        names[i],
        r2_score(x_i, y_i),
        mse,
        mse_norm,
        len(x_i)
    ])


mice_base = pd.DataFrame(test_list)
mice_base.columns=['name','r2','mse','mse_norm','len']

with open('outputs/0617_MICE.pkl', 'wb') as f:
    pickle.dump(val_predictions, f)

# Prepare the input data for BPHMF in R version
The training and validation sets are as same as the EBPMF in this note

In [17]:
taxa_info

,family_x,genus_x,order
59604,Fabaceae,Indigofera,Fabales
97019,Acanthaceae,Ruellia,Lamiales
93426,Bromeliaceae,Quesnelia,Poales
58236,Orobanchaceae,Hyobanche,Lamiales
24125,Achariaceae,Chiangiodendron,Malpighiales
...,...,...,...
95471,Cyperaceae,Rhynchospora,Poales
60116,Colchicaceae,Iphigenia,Liliales
82331,Geraniaceae,Pelargonium,Geraniales
3691,Araceae,Aglaonema,Alismatales


In [18]:


# Assume the following variables already exist:
# R_norm     -> normalized trait matrix
# taxa_info  -> DataFrame with columns: family, genus, order
# names      -> trait names
# val_labels -> validation labels
# vad        -> processed trait matrix

# ---------- 1) Save feature matrix X ----------
os.makedirs('R/BHPMF_input', exist_ok=True)

n_rows, n_cols = R_norm.shape
trait_cols = names

X_df = pd.DataFrame(R_norm, columns=trait_cols)

# Save as CSV; missing values are written as empty cells
X_df.to_csv("R/BHPMF_input/0617_X_matrix.csv", index=False, na_rep="")

taxa_df = taxa_info.copy().reset_index(drop=True)
taxa_df.columns = ['family','genus','order']
# Remove leading/trailing spaces and convert taxonomy columns to strings

# Note that the BHMPF need the specific format for the taxonomy matriax, so we reform the data for input.


for c in ["genus", "family", "order"]:
    taxa_df[c] = taxa_df[c].astype(str).str.strip()

# ---------- 2) Standardize genus -> family mapping ----------
genus_family_map = (
    taxa_df.groupby(["genus", "family"])
    .size()
    .reset_index(name="n")
    .sort_values(["genus", "n"], ascending=[True, False])
    .drop_duplicates(subset=["genus"])
    .set_index("genus")["family"]
)

taxa_df["family_clean"] = taxa_df["genus"].map(genus_family_map)

# ---------- 3) Standardize family -> order mapping ----------
family_order_map = (
    taxa_df.groupby(["family_clean", "order"])
    .size()
    .reset_index(name="n")
    .sort_values(["family_clean", "n"], ascending=[True, False])
    .drop_duplicates(subset=["family_clean"])
    .set_index("family_clean")["order"]
)

taxa_df["order_clean"] = taxa_df["family_clean"].map(family_order_map)

# ---------- 4) Generate hierarchy table for BHPMF ----------
hierarchy_df = pd.DataFrame({
    "id": np.arange(1, len(taxa_df) + 1),
    "genus": taxa_df["genus"],
    "family": taxa_df["family_clean"],
    "order": taxa_df["order_clean"]
})

hierarchy_df.to_csv("R/BHPMF_input/0617_hierarchy_info.csv", index=False)


